In [1]:
from _setup import setup_project_root
PROJECT_ROOT = setup_project_root()
PROJECT_ROOT

WindowsPath('D:/AirPollutionPrediction-CNN-BiLSTM')

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import joblib
import tensorflow as tf
load_model = tf.keras.models.load_model

In [11]:
PROJECT_ROOT = Path.cwd().parent   
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

STATION_SPLIT_DIR = DATA_DIR / "station_split_24havg"   
pm25_scaler = joblib.load(ARTIFACTS_DIR / "pm25_scaler.pkl")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("STATION_SPLIT_DIR exists:", STATION_SPLIT_DIR.exists())
print("Loaded pm25_scaler:", ARTIFACTS_DIR / "pm25_scaler.pkl")


PROJECT_ROOT: d:\AirPollutionPrediction-CNN-BiLSTM
STATION_SPLIT_DIR exists: True
Loaded pm25_scaler: d:\AirPollutionPrediction-CNN-BiLSTM\artifacts\pm25_scaler.pkl


c:\Users\KimNgan\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.5.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [4]:
target_col = "PM2.5_24h_avg"
id_cols = ["Station_No", "date"]

SEQ_LEN = 48
EPOCHS = 200
BATCH = 64
PATIENCE = 20
LR = 3e-4
LSTM_UNITS = 128

print("Target:", target_col)
print("SEQ_LEN:", SEQ_LEN)


Target: PM2.5_24h_avg
SEQ_LEN: 48


In [5]:
import numpy as np
from sklearn.metrics import r2_score

def metrics_real_from_scaled(y_true_scaled, y_pred_scaled, scaler, eps=1e-6, mape_threshold=1.0):
    # Đảm bảo đầu vào là mảng phẳng
    y_true_scaled = np.asarray(y_true_scaled).reshape(-1)
    y_pred_scaled = np.asarray(y_pred_scaled).reshape(-1)

    # Giải mã (Inverse Transform) về đơn vị thực tế (µg/m³)
    yt = scaler.inverse_transform(y_true_scaled.reshape(-1,1)).ravel()
    yp = scaler.inverse_transform(y_pred_scaled.reshape(-1,1)).ravel()

    # Tính toán các chỉ số cơ bản
    diff = yp - yt
    mae = float(np.mean(np.abs(diff)))
    rmse = float(np.sqrt(np.mean(diff**2)))

    # --- TÍNH R^2 SCORE ---
    # R2 = 1 là dự báo hoàn hảo. R2 = 0 là dự báo bằng giá trị trung bình.
    r2 = float(r2_score(yt, yp))

    # Tính MAPE với ngưỡng loại bỏ giá trị nhỏ (tránh chia cho ~0)
    mask = np.abs(yt) >= mape_threshold
    mape = float(np.mean(np.abs(diff[mask]) / np.abs(yt[mask]))) if np.any(mask) else float("nan")

    # Tính Hệ số tương quan Pearson (Correlation)
    if np.std(yt) < eps or np.std(yp) < eps:
        corr = float("nan")
    else:
        corr = float(np.corrcoef(yt, yp)[0,1])

    return {
        "Correlation": corr, 
        "R2": r2,           # Chỉ số mới
        "RMSE": rmse, 
        "MAPE": mape, 
        "MAE": mae
    }

In [6]:
station_dirs = sorted([p for p in STATION_SPLIT_DIR.glob("station_*") if p.is_dir()],
                      key=lambda p: int(p.name.split("_")[1]))

print("Found stations:", [p.name for p in station_dirs])


Found stations: ['station_1', 'station_2', 'station_3', 'station_4', 'station_5', 'station_6']


In [7]:
st_dir = STATION_SPLIT_DIR / "station_1"

train_df = pd.read_csv(st_dir / "train.csv")
val_df   = pd.read_csv(st_dir / "val.csv")
test_df  = pd.read_csv(st_dir / "test.csv")

feature_cols = [c for c in train_df.columns if c not in (id_cols + [target_col])]

X_train = train_df[feature_cols].values.astype(np.float32)
y_train = train_df[target_col].values.astype(np.float32)

X_val = val_df[feature_cols].values.astype(np.float32)
y_val = val_df[target_col].values.astype(np.float32)

X_test = test_df[feature_cols].values.astype(np.float32)
y_test = test_df[target_col].values.astype(np.float32)

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)
print("Num features:", len(feature_cols))


Shapes: (8096, 33) (1734, 33) (1736, 33)
Num features: 33


In [8]:
import numpy as np
import tensorflow as tf
import random
import os

def set_all_seeds(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    # Nếu dùng GPU
    os.environ['TF_DETERMINISTIC_OPS'] = '1'


In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Input, Dropout
def create_sequences(data, seq_length=48):
    """
    Biến dữ liệu từ (N, features) thành (N-seq_length, seq_length, features)
    """
    xs = []
    ys = []
    for i in range(len(data) - seq_length):
        x = data[i:(i + seq_length)]
        y = data[i + seq_length]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

# --- CẤU TRÚC SIMPLE RNN ---
def build_mlp(input_shape):
    # input_shape sẽ là (48, num_features)
    model = Sequential([
        Input(shape=input_shape),
        Flatten(), # Làm phẳng chuỗi 48 bước thành 1 vector duy nhất
        Dense(256, activation='relu'),
        Dropout(0.2),
        Dense(128, activation='relu'),
        Dense(1) # Đầu ra dự báo PM2.5
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

In [14]:
import os
from pathlib import Path

# 0. Tạo thư mục lưu trữ model MLP nếu chưa có
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

all_rows_mlp = []
all_y_true_mlp = []
all_y_pred_mlp = []
all_st_ids_mlp = []

for st_dir in station_dirs:
    st_id = int(st_dir.name.split("_")[1])
    
    # --- 1. Load dữ liệu ---
    train_df = pd.read_csv(st_dir / "train.csv")
    test_df  = pd.read_csv(st_dir / "test.csv")
    
    X_train_raw = train_df[feature_cols].values.astype(np.float32)
    y_train_raw = train_df[target_col].values.astype(np.float32)
    X_test_raw  = test_df[feature_cols].values.astype(np.float32)
    y_test_raw  = test_df[target_col].values.astype(np.float32)

    # --- 2. Tạo dữ liệu 3D ---
    X_train_seq, _ = create_sequences(X_train_raw, SEQ_LEN)
    X_test_seq, _  = create_sequences(X_test_raw, SEQ_LEN)
    y_train_aligned = y_train_raw[SEQ_LEN:]
    y_test_aligned  = y_test_raw[SEQ_LEN:]

    # --- 3. Huấn luyện MLP ---
    input_shape = (X_train_seq.shape[1], X_train_seq.shape[2])
    set_all_seeds(42) # Cố định seed
    mlp_model = build_mlp(input_shape)
    
    print(f"🚀 Training MLP - Station {st_id}...")
    mlp_model.fit(X_train_seq, y_train_aligned, epochs=15, batch_size=32, verbose=0)

    # --- BƯỚC THÊM: LƯU MODEL FILE (.keras) ---
    model_path = ARTIFACTS_DIR / f"mlp_station{st_id}_seq{SEQ_LEN}.keras"
    mlp_model.save(model_path)
    print(f"💾 Đã lưu model: {model_path}")

    # --- 4. Dự báo và Inverse Scale ---
    y_pred_scaled = mlp_model.predict(X_test_seq)
    
    y_p_real = pm25_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).flatten()
    y_t_real = pm25_scaler.inverse_transform(y_test_aligned.reshape(-1, 1)).flatten()
    
    all_y_pred_mlp.append(y_p_real)
    all_y_true_mlp.append(y_t_real)
    all_st_ids_mlp.append(np.full(len(y_t_real), st_id))

    # --- 5. Tính Metric ---
    m = metrics_real_from_scaled(y_test_aligned, y_pred_scaled, pm25_scaler)
    row = {"Station_No": st_id, "n_test": len(y_test_aligned)}
    row.update(m)
    all_rows_mlp.append(row)

# Tạo DataFrame kết quả
results_mlp_df = pd.DataFrame(all_rows_mlp).sort_values("Station_No").reset_index(drop=True)
print(results_mlp_df)

🚀 Training MLP - Station 1...
💾 Đã lưu model: d:\AirPollutionPrediction-CNN-BiLSTM\artifacts\mlp_station1_seq48.keras
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
🚀 Training MLP - Station 2...
💾 Đã lưu model: d:\AirPollutionPrediction-CNN-BiLSTM\artifacts\mlp_station2_seq48.keras
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
🚀 Training MLP - Station 3...
💾 Đã lưu model: d:\AirPollutionPrediction-CNN-BiLSTM\artifacts\mlp_station3_seq48.keras
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
🚀 Training MLP - Station 4...
💾 Đã lưu model: d:\AirPollutionPrediction-CNN-BiLSTM\artifacts\mlp_station4_seq48.keras
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
🚀 Training MLP - Station 5...
💾 Đã lưu model: d:\AirPollutionPrediction-CNN-BiLSTM\artifacts\mlp_station5_seq48.keras
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
🚀 Training MLP - Station 6...
💾 Đã lưu model: d:\AirPollutionPrediction-CNN-BiLSTM\artifacts\mlp_station6_seq48.keras
53/53 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
   Station_No  n_test  Correlation        R2       RMSE   